In [2]:
pip install sqlalchemy pymysql


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymysql

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

In [4]:
df = pd.read_csv("/Users/sreeharivenkataraman/Personal Files/DS_Projects/project_2/online_food_delivery_analysis/dataset/food_delivery_eda_ready.csv")

In [5]:
df.shape

(100000, 34)

In [6]:
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "12345678"

In [7]:
connection = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

cursor = connection.cursor()

In [8]:
cursor.execute(
    "CREATE DATABASE IF NOT EXISTS online_food_delivery"
)

connection.commit()

cursor.close()
connection.close()

In [9]:
encoded_password = quote_plus(MYSQL_PASSWORD)

In [10]:
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{encoded_password}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/online_food_delivery"
)

In [11]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))

    print(result.fetchone())

(1,)


In [12]:
df.dtypes

Order_ID                        str
Customer_ID                     str
Customer_Age                float64
Customer_Gender                 str
City                            str
Area                            str
Restaurant_ID                   str
Restaurant_Name                 str
Cuisine_Type                    str
Order_Date                      str
Order_Time                      str
Delivery_Time_Min           float64
Distance_km                 float64
Order_Value                 float64
Discount_Applied            float64
Final_Amount                float64
Payment_Mode                    str
Order_Status                    str
Cancellation_Reason             str
Delivery_Partner_ID             str
Delivery_Rating             float64
Restaurant_Rating           float64
Order_Day                       str
Peak_Hour                       str
Profit_Margin               float64
Profit_Margin_Percentage    float64
Age_Group                       str
Delivery_Performance        

In [13]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

In [14]:
from sqlalchemy.types import (
    VARCHAR,
    INTEGER,
    FLOAT,
    DATE
)

In [15]:
sql_dtypes = {
    "Order_ID": VARCHAR(20),
    "Customer_ID": VARCHAR(20),
    "Customer_Age": INTEGER(),
    "Customer_Gender": VARCHAR(20),
    "City": VARCHAR(50),
    "Area": VARCHAR(50),

    "Restaurant_ID": VARCHAR(20),
    "Restaurant_Name": VARCHAR(100),
    "Cuisine_Type": VARCHAR(50),

    "Order_Date": DATE(),
    "Order_Time": VARCHAR(20),

    "Delivery_Time_Min": FLOAT(),
    "Distance_km": FLOAT(),

    "Order_Value": FLOAT(),
    "Discount_Applied": FLOAT(),
    "Final_Amount": FLOAT(),

    "Payment_Mode": VARCHAR(30),
    "Order_Status": VARCHAR(30),
    "Cancellation_Reason": VARCHAR(100),

    "Delivery_Partner_ID": VARCHAR(30),

    "Delivery_Rating": FLOAT(),
    "Restaurant_Rating": FLOAT(),

    "Order_Day": VARCHAR(20),
    "Peak_Hour": VARCHAR(20),

    "Profit_Margin": FLOAT(),
    "Profit_Margin_Percentage": FLOAT(),

    "Age_Group": VARCHAR(30),
    "Delivery_Performance": VARCHAR(30),

    "Order_Month": VARCHAR(20),
    "Order_Month_Number": INTEGER(),

    "Is_Cancelled": INTEGER(),

    "Revenue": FLOAT(),
    "Estimated_Profit": FLOAT()
}

In [16]:
df.columns.tolist()

['Order_ID',
 'Customer_ID',
 'Customer_Age',
 'Customer_Gender',
 'City',
 'Area',
 'Restaurant_ID',
 'Restaurant_Name',
 'Cuisine_Type',
 'Order_Date',
 'Order_Time',
 'Delivery_Time_Min',
 'Distance_km',
 'Order_Value',
 'Discount_Applied',
 'Final_Amount',
 'Payment_Mode',
 'Order_Status',
 'Cancellation_Reason',
 'Delivery_Partner_ID',
 'Delivery_Rating',
 'Restaurant_Rating',
 'Order_Day',
 'Peak_Hour',
 'Profit_Margin',
 'Profit_Margin_Percentage',
 'Age_Group',
 'Delivery_Performance',
 'Order_Month',
 'Order_Month_Number',
 'Is_Cancelled',
 'Revenue',
 'Estimated_Profit',
 'Age_Was_Missing']

In [17]:
df.to_sql(
    name="food_delivery_orders",
    con=engine,
    if_exists="replace",
    index=False,
    dtype=sql_dtypes,
    chunksize=5000
)

100000

In [18]:
query = """
SELECT COUNT(*) AS total_rows
FROM food_delivery_orders;
"""

pd.read_sql(query, engine)

,total_rows
0,100000


In [19]:
query = """
SELECT *
FROM food_delivery_orders
LIMIT 5;
"""

pd.read_sql(query, engine)

,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Profit_Margin,Profit_Margin_Percentage,Age_Group,Delivery_Performance,Order_Month,Order_Month_Number,Is_Cancelled,Revenue,Estimated_Profit,Age_Was_Missing
0,ORD000001,CUST6948,19,Male,Unknown,Central,RES936,Restaurant_29,Chinese,2024-10-20,...,0.13,13.0,Young Adult,Slow,October,10,0,1147.0,149.11,0
1,ORD000002,CUST6515,39,Female,Chennai,North,RES689,Restaurant_419,Chinese,2024-08-12,...,0.48,48.0,Unknown,Fast,August,8,0,4849.0,2327.52,1
2,ORD000003,CUST1765,39,Male,Delhi,Unknown,RES723,Restaurant_244,Arabian,2024-12-08,...,0.08,8.0,Unknown,Slow,December,12,0,737.0,58.96,1
3,ORD000004,CUST2744,39,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,2024-10-08,...,0.04,4.0,Unknown,Slow,October,10,1,0.0,0.00,1
4,ORD000005,CUST4389,57,Female,Chennai,South,RES419,Restaurant_262,Chinese,2024-02-04,...,0.12,12.0,Senior Adult,Moderate,February,2,0,352.0,42.24,0


In [20]:
query = """
DESCRIBE food_delivery_orders;
"""

pd.read_sql(query, engine)

,Field,Type,Null,Key,Default,Extra
0,Order_ID,varchar(20),YES,,None,
1,Customer_ID,varchar(20),YES,,None,
2,Customer_Age,int,YES,,None,
3,Customer_Gender,varchar(20),YES,,None,
4,City,varchar(50),YES,,None,
5,Area,varchar(50),YES,,None,
6,Restaurant_ID,varchar(20),YES,,None,
7,Restaurant_Name,varchar(100),YES,,None,
8,Cuisine_Type,varchar(50),YES,,None,
9,Order_Date,date,YES,,None,
